# Land-cover sampling at metro observation points

Install `pip install -c requirements/constraints.txt -e ".[dev,research]"` from the repository root and select that environment as the notebook kernel. This workflow samples all bundled annual LULC rasters with the tested `gobahar.raster` helper. Historical experiments remain in `research/land_cover/Untitled.ipynb`.

**Required:** set `POINTS_CRS` from authoritative metadata. The historical code used EPSG:32641 while other project notebooks used EPSG:32640; the correct reference system is not inferred here. Pixel values retain their source categories. Outside/masked/nodata cells become missing values, never fabricated zeros. A successful run does not validate the chosen CRS or the input data.

In [ ]:
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    path
    for path in candidates
    if (path / "pyproject.toml").is_file() and (path / "research").is_dir()
)

In [ ]:
import os

import pandas as pd

from gobahar.raster import sample_raster

INPUT = ROOT / "research" / "land_cover" / "Metro-Subsidence.xlsx"
RASTER_DIRECTORY = INPUT.parent
OUTPUT = ROOT / "outputs" / "land_cover.csv"
# Replace None with the documented source CRS, e.g. "EPSG:32640" only if verified.
POINTS_CRS = os.environ.get("GOBAHAR_POINTS_CRS") or None
if POINTS_CRS is None:
    raise ValueError("Set POINTS_CRS from source metadata before running this analysis.")

In [ ]:
points = pd.read_excel(INPUT)
coordinates = points[["X_UTM", "Y_UTM"]].to_numpy(dtype=float)
rasters = sorted(RASTER_DIRECTORY.glob("LULC_*_10m.tif"))
if not rasters:
    raise FileNotFoundError(f"No land-cover rasters found in {RASTER_DIRECTORY}")
for raster in rasters:
    column = "_".join(raster.stem.split("_")[:2])
    if column in points.columns:
        raise ValueError(f"Refusing to replace existing column {column}")
    points[column] = sample_raster(raster, coordinates, points_crs=POINTS_CRS)
points.head()

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT.open("x", encoding="utf-8", newline="") as handle:
    points.to_csv(handle, index=False)
print(f"Saved {len(points)} points and {len(rasters)} raster years to {OUTPUT}")
print(points.filter(like="LULC_").isna().sum().rename("missing_samples"))

A repeated run refuses to overwrite the output. Choose a new output name or deliberately remove a prior generated result. Record the source CRS, raster and workbook checksums, environment versions, and acquisition provenance with any result used in a report. Investigate unexpected missing samples before drawing conclusions.